### Task 1: For this analysis, I have chosen Python using the Pandas library within a Jupyter Notebook environment. This allows for efficient handling of the 5.6 million row dataset and provides a clear, reproducible record of the cleaning process.

In [39]:
import pandas as pd
import glob
import os

pd.set_option('display.max_columns', None)

print("Tools are ready!")

Tools are ready!


### Task 2: Data Aggregation
I merged 12 months of individual Cyclistic trip data into a single master dataframe. This ensures that the analysis covers a full year of trends and allows for comparisons across different seasons.

In [40]:
# 1. Path to your raw data
path = '../data_raw' 

# 2. Find all CSV files in that folder
all_files = glob.glob(os.path.join(path, "*.csv"))

# 3. Read and combine them into one 'Master' Dataframe (df)
# This might take a moment—you're loading millions of rows!
df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

print(f"✅ Success! Merged {len(all_files)} months of data.")

✅ Success! Merged 12 months of data.


### Task 3: Data Inspection & Schema Validation
I performed an initial inspection of the merged data. I identified that the date columns were stored as strings and that several columns (start_station_name, end_station_name) contained null values.

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5601662 entries, 0 to 5601661
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 555.6 MB


### Task 4: Data Transformation
To enable analysis, I converted the date strings into datetime objects. I then derived two new columns: 'ride_length' (in minutes) and 'day_of_week' (where 0 is Monday and 6 is Sunday).

In [42]:
# 1. Convert strings to Datetime (this takes about 20-30 seconds)
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# 2. Calculate ride length in minutes
# We divide by 60 because total_seconds() is the raw unit
df['ride_length'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# 3. Calculate day of the week as an integer (0=Monday, 6=Sunday)
df['day_of_week'] = df['started_at'].dt.dayofweek

# 4. Preview the results to verify the ride_length scale
print("--- Task 4 Preview ---")
print(df[['started_at', 'ended_at', 'ride_length', 'day_of_week']].head())

--- Task 4 Preview ---
               started_at                ended_at  ride_length  day_of_week
0 2026-01-31 09:13:09.018 2026-01-31 09:28:10.302    15.021400            5
1 2026-01-15 14:25:42.526 2026-01-15 14:33:18.854     7.605467            3
2 2026-01-06 12:55:33.572 2026-01-06 13:02:17.922     6.739167            1
3 2026-01-26 16:22:25.011 2026-01-26 16:53:15.197    30.836433            0
4 2026-01-10 18:13:30.139 2026-01-10 19:31:56.971    78.447200            5


### Task 5: Data Quality Audit
Before applying any cleaning logic, I performed a null-value audit to identify "holes" in the dataset. This reveals whether data is missing at random or if there are systematic issues with specific columns. This step is crucial for determining whether to delete data or fill it with placeholders.

In [43]:
null_summary = df.isnull().sum()

print("--- Data Quality Audit Results ---")
print(null_summary)

--- Data Quality Audit Results ---
ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1192515
start_station_id      1192515
end_station_name      1254439
end_station_id        1254439
start_lat                   0
start_lng                   0
end_lat                  5742
end_lng                  5742
member_casual               0
ride_length                 0
day_of_week                 0
dtype: int64


### Task 6: Comprehensive Data Cleaning
To ensure data integrity, I performed the following actions:
1. Filled missing 'station' names and IDs with "Unknown" to maintain total ride counts.
2. Removed trips shorter than 60 seconds (potential maintenance or false starts).
3. Removed records with missing coordinate (Lat/Lng) data (approx. 0.1% of the dataset) to ensure visualization accuracy.

In [44]:
# 1. Fill missing station text values to avoid errors in string searching
cols_to_fill = ['start_station_name', 'start_station_id', 'end_station_name', 'end_station_id']
df[cols_to_fill] = df[cols_to_fill].fillna('Unknown')

# 2. Remove Duplicates based on ride_id
df = df.drop_duplicates(subset=['ride_id'])

# 3. Create the Final Filtered Dataframe
# Logic: 
# - Duration >= 1 min (The project guide recommends removing trips < 60 seconds)
# - End Lat is NOT null (Ensures we can map the end destination)
# - Not a test station (Removes internal maintenance logs)
df_final = df[
    (df['ride_length'] >= 1) & 
    (df['end_lat'].notnull()) &
    (~df['start_station_name'].str.contains('test|trial|base', case=False, na=False)) &
    (~df['end_station_name'].str.contains('test|trial|base', case=False, na=False))
].copy()

# 4. Final validation results
print(f"Original Rows: {len(df)}")
print(f"Final Cleaned Rows: {len(df_final)}")
print(f"Rows Removed: {len(df) - len(df_final)}")
print(f"Minimum ride length: {df_final['ride_length'].min():.2f} minutes")

Original Rows: 5601662
Final Cleaned Rows: 5446229
Rows Removed: 155433
Minimum ride length: 1.00 minutes


### Task 7: Final Formatting for Analysis
To prepare for the Analyze phase, I am converting the day of the week from integers to names and setting a categorical sort order. This ensures all future visualizations follow the standard Sunday-to-Saturday sequence automatically.

In [45]:
# 1. Map numbers to Day Names
day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 
           4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df_final['day_of_week'] = df_final['day_of_week'].map(day_map)

# 2. Set Categorical Order (Sunday -> Saturday)
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df_final['day_of_week'] = pd.Categorical(df_final['day_of_week'], categories=day_order, ordered=True)

print("✅ Formatting complete: Days are now names and sorted logically.")

✅ Formatting complete: Days are now names and sorted logically.


### Task 8: Exporting the Processed Dataset
The data cleaning and transformation process is now complete. I am exporting the final, validated dataset to the 'data_processed' folder. This 5.4M row file is the 'Gold' version that will be used for all descriptive analysis and Tableau dashboards.

In [46]:
# Set the output path to your existing folder
output_file = '../data_processed/cyclistic_final_clean.csv'

# Export to CSV 
# index=False ensures we don't save the row numbers as a separate column
df_final.to_csv(output_file, index=False)

print(f"✅ Success! Your clean dataset is now saved in: {output_file}")

✅ Success! Your clean dataset is now saved in: ../data_processed/cyclistic_final_clean.csv
